# IsyaratKu — Latih model kata (`signs`)

Model kata dilatih dari dua **dataset publik** (PRD §11.1, DECISIONS #21, #22, #29):

| Sumber | Isi | Format | Lisensi |
| --- | --- | --- | --- |
| [WL-BISINDO](https://www.kaggle.com/datasets/glennleonali/wl-bisindo) (Kindy, Leonali & Lucky, 2025) | 32 kata × 10 sampel × 5 peraga | video `signer{S}_label{L}_sample{N}.mp4` | CC BY-NC 4.0 |
| [Mendeley BISINDO NMM `yzhzkzbnmw`](https://data.mendeley.com/datasets/yzhzkzbnmw) (Ramadhana dkk., Binus) | 40 kata × 60 sampel | landmark `.npy` (30 × 447) | CC BY 4.0 |

Keduanya diubah ke **30 frame × 126 fitur**, sama persis dengan `src/lib/features.ts`. Label dan urutan output dari
`src/data/vocab.json` (`signs`, 64 kelas termasuk `_IDLE`; nama folder → gloss di `datasetLabels`).

Model (PRD §11.2): GRU 64 (return seq) → GRU 64 → Dense 64 → Softmax 64. GRU memakai `reset_after=False` agar
ekspor manual cocok dengan TF.js (dicek `src/lib/tfModel.test.ts`).

Jalankan di **Google Colab** (Runtime → Run all; GPU tidak wajib). Konversi video WL-BISINDO memakan waktu paling lama
(±20–40 menit); hasilnya di-cache di `data/`.

## 1. Instalasi

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # TF.js reads Keras 2 model configs (DECISIONS #23)
%pip install -q tf_keras scikit-learn matplotlib mediapipe kagglehub opencv-python-headless

## 2. Label (`vocab.json`)

Di Colab, unggah `src/data/vocab.json` saat diminta.

In [ ]:
import glob
import json
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

vocab_path = Path("../src/data/vocab.json")
if not vocab_path.exists():
    vocab_path = DATA_DIR / "vocab.json"
    if not vocab_path.exists():
        print("Pilih file src/data/vocab.json")
        vocab_path.write_bytes(next(iter(files.upload().values())))
VOCAB = json.loads(vocab_path.read_text(encoding="utf-8"))
SIGNS = VOCAB["signs"]            # output index = position in this list; never reorder
IDLE = "_IDLE"
WL_NAMES = list(VOCAB["datasetLabels"]["wl-bisindo"].values())       # WL label number -> gloss
MENDELEY_MAP = VOCAB["datasetLabels"]["mendeley-yzhzkzbnmw"]          # folder/file prefix -> gloss
assert SIGNS[-1] == IDLE and len(WL_NAMES) == 32 and len(MENDELEY_MAP) == 40
print(len(SIGNS), "kelas")

## 3. Fitur

Salinan `normalize_hand` / `to_features` dari `src/lib/features.ts` (sama dengan `train_letters.ipynb`).
**Panjang window = 30 frame.** Video di-*resample* ke `TARGET_FPS` (laju deteksi aplikasi di HP) sebelum diambil
landmark-nya, dan augmentasi *time-warp* membuat model tahan terhadap perbedaan kecepatan (DECISIONS #22).

In [ ]:
import math
import numpy as np

LANDMARKS, PER_HAND = 21, 63
FEATURES = 126
WINDOW = 30
TARGET_FPS = 15  # approximate detection rate of the app on a mid-range phone


def normalize_hand(points, aspect=1.0):
    """Python port of normalizeHand() in src/lib/features.ts. Keep both in sync."""
    wrist, ref = points[0], points[9]
    size = math.sqrt(((ref[0] - wrist[0]) * aspect) ** 2 + (ref[1] - wrist[1]) ** 2
                     + ((ref[2] - wrist[2]) * aspect) ** 2)
    if not size > 1e-6:
        return [0.0] * PER_HAND
    out = []
    for x, y, z in points:
        out += [(x - wrist[0]) * aspect / size, (y - wrist[1]) / size, (z - wrist[2]) * aspect / size]
    return out


def to_features(left, right, aspect=1.0):
    return (normalize_hand(left, aspect) if left else [0.0] * PER_HAND) + \
           (normalize_hand(right, aspect) if right else [0.0] * PER_HAND)


def resample(seq, n=WINDOW):
    """Uniformly resample a (T, 126) sequence to n frames."""
    seq = np.asarray(seq, dtype=np.float32)
    idx = np.linspace(0, len(seq) - 1, n).round().astype(int)
    return seq[idx]


_f = normalize_hand([(0.5 + 0.01 * i, 0.5 - 0.02 * i, 0.001 * i) for i in range(LANDMARKS)], aspect=4 / 3)
assert _f[:3] == [0.0, 0.0, 0.0] and abs(math.hypot(*_f[27:30]) - 1) < 1e-9
assert resample(np.arange(10)[:, None].repeat(FEATURES, 1)).shape == (WINDOW, FEATURES)
print("fitur OK")

## 4. WL-BISINDO (video → landmark)

Diunduh lewat `kagglehub` (jika gagal: unduh zip dari Kaggle — perlu akun — lalu unggah). Tiap video:
frame diambil pada `TARGET_FPS`, MediaPipe HandLandmarker (model yang sama dengan aplikasi) → 126 fitur/frame.
Bagian isyarat = dari frame pertama sampai terakhir yang ada tangannya, di-resample ke 30 frame.
Frame sebelum/sesudah isyarat (tangan belum terlihat) menjadi contoh `_IDLE`.
Kiri/kanan memakai label MediaPipe apa adanya, sama dengan `toHandFrame()` (DECISIONS #10).

In [ ]:
import re
import urllib.request
import zipfile

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

WL_CACHE = DATA_DIR / "wl-bisindo.npz"
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/latest/hand_landmarker.task")
if not Path("hand_landmarker.task").exists():
    urllib.request.urlretrieve(MODEL_URL, "hand_landmarker.task")
landmarker = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO, num_hands=2))
_clock_ms = 0  # VIDEO mode needs increasing timestamps across all videos


def frame_features(rgb, t_ms):
    result = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), t_ms)
    hands, spare = {"Left": None, "Right": None}, []
    for points, cats in zip(result.hand_landmarks, result.handedness):
        pts = [(p.x, p.y, p.z) for p in points]
        side = cats[0].category_name if cats else None
        if side in hands and hands[side] is None:
            hands[side] = pts
        else:
            spare.append(pts)
    for pts in spare:
        side = "Left" if hands["Left"] is None else "Right" if hands["Right"] is None else None
        if side:
            hands[side] = pts
    return to_features(hands["Left"], hands["Right"], aspect=rgb.shape[1] / rgb.shape[0])


def video_features(path):
    global _clock_ms
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, round(fps / TARGET_FPS))
    feats, i = [], 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        if i % step == 0:
            _clock_ms += int(1000 / TARGET_FPS)
            feats.append(frame_features(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), _clock_ms))
        i += 1
    cap.release()
    return np.asarray(feats, dtype=np.float32)


if WL_CACHE.exists():
    wl = dict(np.load(WL_CACHE))
    print("Memakai cache WL-BISINDO:", wl["X"].shape)
else:
    try:
        import kagglehub
        wl_root = Path(kagglehub.dataset_download("glennleonali/wl-bisindo"))
    except Exception as err:  # noqa: BLE001 - any failure falls back to a manual upload
        print("kagglehub gagal:", err, "-> unggah zip WL-BISINDO")
        zip_name = next(iter(files.upload())) if IN_COLAB else sorted(glob.glob("../datasets/wl-bisindo/*.zip"))[0]
        wl_root = Path("wl-bisindo")
        zipfile.ZipFile(zip_name).extractall(wl_root)

    pattern = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)", re.I)
    videos = sorted(p for p in wl_root.rglob("*") if p.suffix.lower() in {".mp4", ".mov", ".avi"} and pattern.search(p.stem))
    print("Video ditemukan:", len(videos))
    assert videos, "Nama file tidak dikenali; cek isi wl_root."
    X, y, signer, X_idle, signer_idle, skipped = [], [], [], [], [], 0
    for n, path in enumerate(videos):
        s, label, _ = map(int, pattern.search(path.stem).groups())
        seq = video_features(path)
        has = np.where(np.abs(seq).sum(axis=1) > 0)[0]
        if len(has) < 3:
            skipped += 1
            continue
        X.append(resample(seq[has[0]:has[-1] + 1]))
        y.append(SIGNS.index(WL_NAMES[label]))
        signer.append(s)
        rest = np.concatenate([seq[:has[0]], seq[has[-1] + 1:]])
        if len(rest) >= 8:  # enough "no sign" frames before/after the sign
            X_idle.append(resample(rest))
            signer_idle.append(s)
        if (n + 1) % 100 == 0:
            print(f"  {n + 1}/{len(videos)} video")
    wl = {"X": np.stack(X), "y": np.array(y), "signer": np.array(signer),
          "X_idle": np.stack(X_idle) if X_idle else np.zeros((0, WINDOW, FEATURES), np.float32),
          "signer_idle": np.array(signer_idle)}
    np.savez_compressed(WL_CACHE, **wl)
    print(f"WL-BISINDO: {len(X)} isyarat, {len(X_idle)} _IDLE, {skipped} video tanpa tangan dilewati")

## 5. Mendeley BISINDO NMM (landmark `.npy`)

Zip diunduh langsung (tanpa login). Tiap file: (30, 447) = 149 titik × xyz. Titik 107–127 = tangan kiri subjek,
128–148 = tangan kanan; tangan tak terlihat = 0 (DECISIONS #29). Data sudah dinormalisasi ke bahu, tetapi x/y masih
dalam satuan gambar, jadi **rasio aspek video ditaksir dari data**: aspek yang benar membuat panjang tulang tangan
paling stabil saat tangan berputar.

In [ ]:
import shutil
import urllib.error

MENDELEY_CACHE = DATA_DIR / "mendeley.npz"
MENDELEY_ZIP = "https://data.mendeley.com/public-api/zip/yzhzkzbnmw/download/3"
LEFT = slice(107 * 3, 128 * 3)
RIGHT = slice(128 * 3, 149 * 3)
BONES = [(0, 5), (0, 9), (0, 13), (0, 17), (5, 9), (9, 13), (13, 17)]


def bone_spread(samples, aspect):
    """Mean coefficient of variation of bone lengths over frames (lower = geometry more consistent)."""
    cvs = []
    for arr in samples:
        for block in (LEFT, RIGHT):
            pts = arr[:, block].reshape(len(arr), LANDMARKS, 3)
            pts = pts[np.abs(pts).sum(axis=(1, 2)) > 0]
            if len(pts) < 5:
                continue
            scaled = pts * np.array([aspect, 1.0, aspect])
            lengths = np.stack([np.linalg.norm(scaled[:, a] - scaled[:, b], axis=1) for a, b in BONES], axis=1)
            cvs.append((lengths.std(axis=0) / (lengths.mean(axis=0) + 1e-9)).mean())
    return float(np.mean(cvs))


if MENDELEY_CACHE.exists():
    md_data = dict(np.load(MENDELEY_CACHE))
    print("Memakai cache Mendeley:", md_data["X"].shape, "aspek", float(md_data["aspect"]))
else:
    if not Path("mendeley.zip").exists():
        try:
            urllib.request.urlretrieve(MENDELEY_ZIP, "mendeley.zip")
        except urllib.error.HTTPError as err:
            # Mendeley sometimes refuses scripted downloads (403). Fallback: the zip downloaded by
            # hand from https://data.mendeley.com/datasets/yzhzkzbnmw ("Download All"), put in
            # Google Drive as MyDrive/mendeley.zip, or uploaded here.
            print(f"Unduhan otomatis ditolak ({err.code}). Memakai mendeley.zip dari Google Drive.")
            drive_zip = Path("/content/drive/MyDrive/mendeley.zip")
            if IN_COLAB and not drive_zip.exists():
                from google.colab import drive  # type: ignore
                drive.mount("/content/drive")
            if drive_zip.exists():
                shutil.copy(drive_zip, "mendeley.zip")
            elif IN_COLAB:
                print("MyDrive/mendeley.zip tidak ada. Pilih file zip Mendeley untuk diunggah.")
                Path("mendeley.zip").write_bytes(next(iter(files.upload().values())))
            else:
                raise RuntimeError("Unduh zip dari halaman Mendeley, simpan sebagai mendeley.zip") from err
    zipfile.ZipFile("mendeley.zip").extractall("mendeley")
    # The download may contain nested zips; extract them too.
    for inner in Path("mendeley").rglob("*.zip"):
        zipfile.ZipFile(inner).extractall(inner.with_suffix(""))
    files_npy = sorted(Path("mendeley").rglob("*.npy"))
    print("File .npy:", len(files_npy))
    raw, labels, unknown = [], [], set()
    for p in files_npy:
        prefix = re.sub(r"_\d+$", "", p.stem)
        gloss = MENDELEY_MAP.get(prefix)
        if gloss is None:
            unknown.add(prefix)
            continue
        arr = np.load(p)
        assert arr.shape == (WINDOW, 447), (p, arr.shape)
        raw.append(arr)
        labels.append(SIGNS.index(gloss))
    print("Prefix tak dikenal:", sorted(unknown) or "-")

    candidates = np.round(np.arange(0.5, 2.51, 0.05), 2)
    rng_pick = np.random.default_rng(0)
    probe = [raw[i] for i in rng_pick.choice(len(raw), size=min(300, len(raw)), replace=False)]
    spreads = [bone_spread(probe, a) for a in candidates]
    aspect = float(candidates[int(np.argmin(spreads))])
    print(f"Aspek taksiran: {aspect} (16/9 = {16 / 9:.2f})")

    def to_seq(arr):
        out = []
        for frame in arr:
            l, r = frame[LEFT].reshape(LANDMARKS, 3), frame[RIGHT].reshape(LANDMARKS, 3)
            out.append(to_features(l.tolist() if np.abs(l).sum() > 0 else None,
                                   r.tolist() if np.abs(r).sum() > 0 else None, aspect))
        return out

    md_data = {"X": np.asarray([to_seq(a) for a in raw], np.float32), "y": np.array(labels), "aspect": aspect}
    np.savez_compressed(MENDELEY_CACHE, **md_data)
    print("Mendeley:", md_data["X"].shape)

## 6. Split

- **Uji:** WL-BISINDO peraga 4 (*signer-independent*, sesuai protokol paper) + 15% Mendeley (stratified).
- **Validasi:** 15% dari sisa. **Latih:** sisanya.
- `_IDLE`: dari awal/akhir video WL (ikut split peraga) + window tanpa tangan sama sekali.

**Catatan laporan:** Mendeley tidak mencantumkan ID peraga, jadi bagian ujinya bisa memuat peraga yang sama dengan data latih.

In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)
IDLE_ID = SIGNS.index(IDLE)
TEST_SIGNER = 4

wl_test = wl["signer"] == TEST_SIGNER
idle_test = wl["signer_idle"] == TEST_SIGNER
md_tr, md_te, ymd_tr, ymd_te = train_test_split(md_data["X"], md_data["y"], test_size=0.15,
                                                random_state=SEED, stratify=md_data["y"])
blank = np.zeros((80, WINDOW, FEATURES), np.float32)  # "no hands in view" windows

X_rest = np.concatenate([wl["X"][~wl_test], md_tr, wl["X_idle"][~idle_test], blank[:60]])
y_rest = np.concatenate([wl["y"][~wl_test], ymd_tr, np.full((~idle_test).sum(), IDLE_ID), np.full(60, IDLE_ID)])
X_test = np.concatenate([wl["X"][wl_test], md_te, wl["X_idle"][idle_test], blank[60:]])
y_test = np.concatenate([wl["y"][wl_test], ymd_te, np.full(idle_test.sum(), IDLE_ID), np.full(20, IDLE_ID)])
source_test = np.array(["wl"] * wl_test.sum() + ["mendeley"] * len(md_te) + ["idle"] * (idle_test.sum() + 20))

X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.15, random_state=SEED, stratify=y_rest)
print(f"train={len(X_train)} val={len(X_val)} test={len(X_test)}")
missing = [SIGNS[i] for i in range(len(SIGNS)) if i not in set(y_train.tolist())]
assert not missing, f"Kelas tanpa data latih: {missing}"

## 7. Augmentasi (PRD §11.2 + time-warp)

Per sampel (sama untuk semua frame): rotasi ±15° per tangan, skala ±10%, mirror (tukar kiri/kanan + balik x); per frame:
noise Gaussian kecil. Ditambah **time-warp** ×0,75–1,33 (isyarat lebih cepat/lambat) karena laju deteksi HP berbeda-beda.
Tangan kosong tetap 0.

In [ ]:
AUG_COPIES = 4


def augment_seq(seq):
    seq = seq.copy()
    if rng.random() < 0.5:  # mirror
        left, right = seq[:, :PER_HAND].copy(), seq[:, PER_HAND:].copy()
        seq[:, :PER_HAND], seq[:, PER_HAND:] = right, left
        seq[:, 0::3] *= -1
    for block in (slice(0, PER_HAND), slice(PER_HAND, FEATURES)):
        pts = seq[:, block].reshape(WINDOW, LANDMARKS, 3)
        present = np.abs(pts).sum(axis=(1, 2)) > 0
        a = np.deg2rad(rng.uniform(-15, 15))
        c, s = np.cos(a), np.sin(a)
        x, y = pts[..., 0].copy(), pts[..., 1].copy()
        pts[..., 0], pts[..., 1] = c * x - s * y, s * x + c * y
        pts *= rng.uniform(0.9, 1.1)
        pts += rng.normal(0, 0.02, pts.shape)
        pts[:, 0] = 0.0          # wrist stays at the origin
        pts[~present] = 0.0      # missing hand stays missing
        seq[:, block] = pts.reshape(WINDOW, PER_HAND)
    speed = rng.uniform(0.75, 1.33)  # time-warp: resample then crop/pad back to WINDOW
    warped = resample(seq, max(8, int(round(WINDOW / speed))))
    if len(warped) >= WINDOW:
        start = rng.integers(0, len(warped) - WINDOW + 1)
        return warped[start:start + WINDOW]
    pad = WINDOW - len(warped)
    left = rng.integers(0, pad + 1)
    return np.concatenate([np.repeat(warped[:1], left, 0), warped, np.repeat(warped[-1:], pad - left, 0)])


X_aug = [X_train] + [np.stack([augment_seq(s) for s in X_train]) for _ in range(AUG_COPIES)]
X_train_aug = np.concatenate(X_aug).astype(np.float32)
y_train_aug = np.tile(y_train, AUG_COPIES + 1)
assert X_train_aug.shape[1:] == (WINDOW, FEATURES)
print("train setelah augmentasi:", X_train_aug.shape)

## 8. Latih model

In [ ]:
import tf_keras as keras
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(SEED)
model = keras.Sequential([
    keras.layers.Input(shape=(WINDOW, FEATURES)),
    keras.layers.GRU(64, return_sequences=True, reset_after=False),
    keras.layers.GRU(64, reset_after=False),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(len(SIGNS), activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

classes = np.unique(y_train_aug)
weights = dict(zip(classes, compute_class_weight("balanced", classes=classes, y=y_train_aug)))
history = model.fit(X_train_aug, y_train_aug, validation_data=(X_val, y_val), epochs=120, batch_size=64,
                    class_weight=weights, verbose=2,
                    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)])

## 9. Evaluasi (untuk laporan)

Target PRD: akurasi dan macro-F1 kata ≥ 80%.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score

REPORT_DIR = Path("report")
REPORT_DIR.mkdir(exist_ok=True)
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
acc = float((y_pred == y_test).mean())
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Akurasi uji: {acc:.3f} | macro-F1: {macro_f1:.3f} ({len(y_test)} sampel)")
for src in ("wl", "mendeley", "idle"):
    m = source_test == src
    print(f"  {src:9s}: {(y_pred[m] == y_test[m]).mean():.3f} ({m.sum()} sampel)")

present = sorted(set(y_test.tolist()) | set(y_pred.tolist()))
report = classification_report(y_test, y_pred, labels=present, target_names=[SIGNS[i] for i in present], zero_division=0)
print(report)
(REPORT_DIR / "signs_classification_report.txt").write_text(
    f"accuracy {acc:.4f}\nmacro_f1 {macro_f1:.4f}\n\n" + report)
cm = confusion_matrix(y_test, y_pred, labels=list(range(len(SIGNS))))
np.savetxt(REPORT_DIR / "signs_confusion_matrix.csv", cm, fmt="%d", delimiter=",", header=",".join(SIGNS), comments="")
fig, ax = plt.subplots(figsize=(18, 18))
ConfusionMatrixDisplay(cm, display_labels=SIGNS).plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=90)
ax.set_title(f"Confusion matrix kata (akurasi {acc:.1%}, macro-F1 {macro_f1:.1%})")
fig.savefig(REPORT_DIR / "signs_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history["accuracy"], label="train")
ax.plot(history.history["val_accuracy"], label="val")
ax.set_xlabel("epoch"); ax.set_ylabel("akurasi"); ax.legend()
fig.savefig(REPORT_DIR / "signs_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Ekspor ke TF.js

Format *layers-model* ditulis sendiri (DECISIONS #23). Dicek dengan forward pass NumPy GRU dari file ekspor.
Salin isi folder `signs/` ke `public/models/signs/`; toggle **Kata** lalu bisa diaktifkan di Fase 5.

In [ ]:
import shutil


def save_tfjs_layers_model(model, out_dir):
    """Write a TF.js layers-model (model.json + group1-shard1of1.bin) from a tf_keras model."""
    out_dir = Path(out_dir)
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True)
    specs, chunks = [], []
    for layer in model.layers:
        for w in layer.weights:
            # TF.js looks weights up as "<layer name>/<weight name>", e.g. "gru/recurrent_kernel".
            name = f"{layer.name}/{w.name.split('/')[-1].split(':')[0]}"
            value = np.asarray(w.numpy(), dtype="<f4")
            specs.append({"name": name, "shape": list(value.shape), "dtype": "float32"})
            chunks.append(value.tobytes())
    (out_dir / "group1-shard1of1.bin").write_bytes(b"".join(chunks))
    (out_dir / "model.json").write_text(json.dumps({
        "format": "layers-model",
        "generatedBy": f"tf_keras {keras.__version__}",
        "convertedBy": "IsyaratKu save_tfjs_layers_model",
        "modelTopology": json.loads(model.to_json()),
        "weightsManifest": [{"paths": ["group1-shard1of1.bin"], "weights": specs}],
    }))
    return specs


OUT_DIR = Path("signs")
specs = save_tfjs_layers_model(model, OUT_DIR)
assert model.output_shape[-1] == len(SIGNS)

# Check: NumPy forward pass (Keras GRU, reset_after=False) using ONLY the exported files.
buf = np.frombuffer((OUT_DIR / "group1-shard1of1.bin").read_bytes(), dtype="<f4")
P, offset = {}, 0
for spec in specs:
    n = int(np.prod(spec["shape"]))
    P[spec["name"]] = buf[offset:offset + n].reshape(spec["shape"])
    offset += n
sig = lambda v: 1 / (1 + np.exp(-v))


def gru(x, name, return_sequences):
    W, U, b = P[f"{name}/kernel"], P[f"{name}/recurrent_kernel"], P[f"{name}/bias"]
    u = U.shape[0]
    h = np.zeros((x.shape[0], u), np.float32)
    outs = []
    for t in range(x.shape[1]):
        xw = x[:, t] @ W + b
        z = sig(xw[:, :u] + h @ U[:, :u])
        r = sig(xw[:, u:2 * u] + h @ U[:, u:2 * u])
        cand = np.tanh(xw[:, 2 * u:] + (r * h) @ U[:, 2 * u:])
        h = z * h + (1 - z) * cand
        outs.append(h)
    return np.stack(outs, 1) if return_sequences else h


sample = X_test[:16]
h = sample
for layer in model.layers:
    if isinstance(layer, keras.layers.GRU):
        h = gru(h, layer.name, layer.return_sequences)
    elif isinstance(layer, keras.layers.Dense):
        h = h @ P[f"{layer.name}/kernel"] + P[f"{layer.name}/bias"]
        if layer.get_config()["activation"] == "relu":
            h = np.maximum(h, 0)
        else:
            h = np.exp(h - h.max(axis=1, keepdims=True)); h = h / h.sum(axis=1, keepdims=True)
diff = np.abs(h - model.predict(sample, verbose=0)).max()
assert diff < 1e-4, diff
print(f"Ekspor OK (selisih maks {diff:.1e}):", sorted(p.name for p in OUT_DIR.iterdir()))

if IN_COLAB:
    shutil.make_archive("signs-model", "zip", OUT_DIR)
    shutil.make_archive("signs-report", "zip", REPORT_DIR)
    files.download("signs-model.zip")
    files.download("signs-report.zip")